# 07. 기본 RAG 파이프라인 구현

지금까지 배운 모든 단계를 연결하여 완성된 RAG 시스템을 구현합니다.

```
【 인덱싱 】
문서 로딩 → 텍스트 분할 → 임베딩 → 벡터 스토어

【 질의응답 】
질문 → Retriever → 컨텍스트 구성 → LLM → 답변
```

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

## Step 1. 인덱싱 - 문서 준비

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. 문서 로딩
loader = TextLoader("data/ai_basic.txt", encoding="utf-8")
docs = loader.load()
print(f"문서 수: {len(docs)}")

# 2. 텍스트 분할
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"청크 수: {len(chunks)}")

# 3. 임베딩 + 벡터 스토어
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 4. 벡터 스토어 생성
vector_store = FAISS.from_documents(chunks, embeddings)
print("벡터 스토어 생성 완료")

문서 수: 1
청크 수: 10
벡터 스토어 생성 완료


## Step 2. 기본 RAG 체인 구성

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 2. 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 질문에 친절하게 답변하는 AI 어시스턴트입니다.
아래 문서를 참고하여 질문에 답하세요.
문서에 없는 내용은 '문서에서 찾을 수 없습니다'라고 답하세요.

[문서]
{context}"""),
    ("human", "{question}")
])

# 3. LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# k개 문서를 하나로 합치는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. RAG 체인 구성
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG 체인 구성 완료")

RAG 체인 구성 완료


In [1]:
docs = [{"page_content": "문서1 내용"}, {"page_content": "문서2 내용"}, {"page_content": "문서3 내용"}]
"\n\n".join(doc["page_content"] for doc in docs)

'문서1 내용\n\n문서2 내용\n\n문서3 내용'

## Step 3. 질의응답 테스트

In [4]:
question = "RAG란 무엇이고, 왜 필요한가요?"

answer = rag_chain.invoke(question)
print(f"Q: {question}")
print(f"A: {answer}")

Q: RAG란 무엇이고, 왜 필요한가요?
A: RAG(Retrieval-Augmented Generation)는 검색(Retrieval)과 생성(Generation)을 결합한 기술로, LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. RAG는 최신 정보를 제공하고, 할루시네이션을 감소시키며, 출처를 제공하는 등의 장점이 있습니다. 이러한 이유로 RAG는 보다 정확하고 신뢰할 수 있는 정보를 제공하는 데 필요합니다.


In [5]:
question = "트랜스포머(Transformer)란 무엇인가요?"

answer = rag_chain.invoke(question)
print(f"Q: {question}")
print(f"A: {answer}")

Q: 트랜스포머(Transformer)란 무엇인가요?
A: 트랜스포머(Transformer)는 2017년 구글이 발표한 신경망 구조로, 자연어 처리 분야에 혁신을 가져온 모델입니다. 어텐션 메커니즘(Attention Mechanism)을 핵심으로 사용하며, 병렬 처리가 가능해 학습 속도가 빠릅니다. BERT, GPT 등 현대 대규모 언어 모델의 기반이 됩니다.


In [6]:
# 문서에 없는 내용 질문
question = "오늘 주식 시장은 어땠나요?"

answer = rag_chain.invoke(question)
print(f"Q: {question}")
print(f"A: {answer}")

Q: 오늘 주식 시장은 어땠나요?
A: 문서에서 찾을 수 없습니다.


## Step 4. 출처(Source) 포함 답변

In [7]:
from langchain_core.runnables import RunnableParallel

# 답변과 출처를 함께 반환하는 체인
rag_chain_with_source = RunnableParallel(
    answer=rag_chain,
    source_documents=retriever
)

result = rag_chain_with_source.invoke("LangChain이 무엇인가요?")

print("[답변]")
print(result["answer"])

print("\n[참고 문서]")
for i, doc in enumerate(result["source_documents"]):
    print(f"  [{i+1}] {doc.metadata.get('source', 'unknown')}")
    print(f"       {doc.page_content[:100]}...")

[답변]
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 이 프레임워크는 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공합니다. LangChain은 Python과 JavaScript 버전이 있으며, RAG, 에이전트, 챗봇 등 다양한 애플리케이션 개발에 활용됩니다.

[참고 문서]
  [1] data/ai_basic.txt
       LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공...
  [2] data/ai_basic.txt
       대규모 언어 모델(LLM)이란?
대규모 언어 모델(Large Language Model, LLM)은 방대한 텍스트 데이터로 학습된 트랜스포머 기반 모델입니다. GPT-4, Clau...
  [3] data/ai_basic.txt
       합성곱 신경망(CNN, Convolutional Neural Network)은 이미지 처리에 특화된 신경망 구조입니다. 필터를 사용해 이미지의 특징을 추출하며, 이미지 분류, 객체...


## Step 5. 스트리밍 답변

In [8]:
question = "딥러닝과 머신러닝의 차이는 무엇인가요?"

print(f"Q: {question}")
print("A: ", end="")

for chunk in rag_chain.stream(question):
    print(chunk, end="", flush=True)

Q: 딥러닝과 머신러닝의 차이는 무엇인가요?
A: 문서에서 찾을 수 없습니다.

## 전체 파이프라인 정리

```
TextLoader          # 문서 로딩
    ↓
RecursiveCharacterTextSplitter  # 청크 분할
    ↓
OpenAIEmbeddings    # 임베딩
    ↓
FAISS               # 벡터 스토어
    ↓
as_retriever()      # 검색
    ↓
ChatPromptTemplate  # 프롬프트 구성
    ↓
ChatOpenAI          # 답변 생성
    ↓
StrOutputParser     # 문자열 출력
```

---

## Naive RAG의 한계

| 한계 | 원인 |
|---|---|
| 검색 품질 편차 | 단일 쿼리로 검색 → 표현 방식에 따라 결과 다름 |
| 관련 없는 문서 포함 | 임계값 없이 k개 무조건 반환 |
| 긴 컨텍스트 노이즈 | 모든 청크를 그대로 LLM에 전달 |

→ **다음 파트(Advanced RAG)** 에서 이 한계를 해결합니다.